In [1]:
import re
import ipaddress
from pathlib import Path
from urllib.parse import unquote

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

try:
    import xlsxwriter                          
    EXCEL_ENGINE = "xlsxwriter"
except ImportError:
    EXCEL_ENGINE = "openpyxl"              

In [2]:
BASE_DIR   = Path(r"C:\Users\LAP TECH\Desktop\DEPI final Project")
FILE_PATH  = BASE_DIR / "tokenized_access_logs.csv"
OUTPUT_DIR = BASE_DIR / "files"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)      

df = pd.read_csv(FILE_PATH)
rows_before, cols_before = df.shape
print("Shape before cleaning:", df.shape)
df.head(3)

Shape before cleaning: (469977, 8)


,Product,Category,Date,Month,Hour,Department,ip,url
0,adidas Brazuca 2017 Official Match Ball,baseball & softball,9/1/2017 6:00,Sep,6,fitness,37.97.182.65,/department/fitness/category/baseball%20&%20so...
1,The North Face Women's Recon Backpack,hunting & shooting,9/1/2017 6:00,Sep,6,fan shop,206.56.112.1,/department/fan%20shop/category/hunting%20&%20...
2,adidas Kids' RG III Mid Football Cleat,featured shops,9/1/2017 6:00,Sep,6,apparel,215.143.180.0,/department/apparel/category/featured%20shops/...


In [3]:
df.columns = [c.strip().lower() for c in df.columns]
df = df.rename(columns={"date": "timestamp"})
print(df.columns.tolist())

['product', 'category', 'timestamp', 'month', 'hour', 'department', 'ip', 'url']


In [4]:
text_cols = df.select_dtypes(include=["object", "string"]).columns

for c in text_cols:
    df[c] = df[c].str.strip().str.replace(r"\s+", " ", regex=True)

In [5]:
parts = df["url"].str.extract(
    r"^/department/(?P<department>.+?)/category/(?P<category>.+?)/product/(?P<product>.+?)(?P<cart>/add_to_cart)?$"
)
assert parts["department"].notna().all(), "في url مش بنفس الشكل المتوقع"

def decode(series):
    mapping = {v: re.sub(r"\s+", " ", unquote(v)).strip() for v in series.unique()}
    return series.map(mapping)

df["action"] = np.where(parts["cart"].notna(), "add_to_cart", "view")

assert (decode(parts["department"]) == df["department"]).all(), "department مختلف عن الـ url"
assert (decode(parts["product"]) == df["product"]).all(), "product مختلف عن الـ url"
url_category = decode(parts["category"])
assert (url_category.str.replace("/", " ") == df["category"]).all(), "category مختلف عن الـ url"

df["category"] = url_category          
print(df["action"].value_counts())

action
view           337057
add_to_cart    132920
Name: count, dtype: int64


In [6]:
def smart_title(text):
    """men's golf clubs -> Men's Golf Clubs  (و indoor/outdoor -> Indoor/Outdoor)"""
    return re.sub(r"[A-Za-z][A-Za-z']*", lambda m: m.group(0).capitalize(), text)

for col in ["department", "category"]:
    mapping = {v: smart_title(v) for v in df[col].unique()}
    df[col] = df[col].map(mapping)

df["category"] = df["category"].replace({
    "Boxing & Mma":   "Boxing & MMA",
    "As Seen On Tv!": "As Seen on TV!",
})

sorted(df["category"].unique())

['Accessories',
 'As Seen on TV!',
 'Baseball & Softball',
 'Basketball',
 'Boxing & MMA',
 'Camping & Hiking',
 'Cardio Equipment',
 'Cleats',
 'Electronics',
 'Featured Shops',
 'Fishing',
 'Fitness Accessories',
 "Girls' Apparel",
 'Golf Apparel',
 'Golf Bags & Carts',
 'Golf Balls',
 'Golf Gloves',
 'Golf Shoes',
 'Hockey',
 'Hunting & Shooting',
 'Indoor/Outdoor Games',
 "Kids' Golf Clubs",
 'Lacrosse',
 "Men's Footwear",
 "Men's Golf Clubs",
 'Shop By Sport',
 'Soccer',
 'Strength Training',
 'Tennis & Racquet',
 'Trade-In',
 'Water Sports',
 "Women's Apparel",
 "Women's Golf Clubs"]

In [7]:
df["timestamp"] = pd.to_datetime(df["timestamp"], format="%m/%d/%Y %H:%M")

assert (df["timestamp"].dt.strftime("%b") == df["month"]).all(), "month مش مطابق للتاريخ"
assert (df["timestamp"].dt.hour == df["hour"]).all(), "hour مش مطابق للتاريخ"

df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)
print(df["timestamp"].min(), "→", df["timestamp"].max())

2017-09-01 06:00:00 → 2018-01-31 23:58:00


In [8]:
df["date"]       = df["timestamp"].dt.normalize()
df["year"]       = df["timestamp"].dt.year
df["month_num"]  = df["timestamp"].dt.month
df["month_name"] = df["timestamp"].dt.month_name()
df["day_name"]   = df["timestamp"].dt.day_name()

ip_is_private = {ip: ipaddress.ip_address(ip).is_private for ip in df["ip"].unique()}
df["is_private_ip"] = df["ip"].map(ip_is_private)

print("Private-IP rows:", int(df["is_private_ip"].sum()))

Private-IP rows: 4260


In [9]:
df = df.drop(columns=["month", "url"])

df = df[[
    "timestamp", "date", "year", "month_num", "month_name", "day_name", "hour",
    "department", "category", "product", "action", "ip", "is_private_ip",
]]
df.head()

,timestamp,date,year,month_num,month_name,day_name,hour,department,category,product,action,ip,is_private_ip
0,2017-09-01 06:00:00,2017-09-01,2017,9,September,Friday,6,Fitness,Baseball & Softball,adidas Brazuca 2017 Official Match Ball,view,37.97.182.65,False
1,2017-09-01 06:00:00,2017-09-01,2017,9,September,Friday,6,Fan Shop,Hunting & Shooting,The North Face Women's Recon Backpack,view,206.56.112.1,False
2,2017-09-01 06:00:00,2017-09-01,2017,9,September,Friday,6,Apparel,Featured Shops,adidas Kids' RG III Mid Football Cleat,view,215.143.180.0,False
3,2017-09-01 06:00:00,2017-09-01,2017,9,September,Friday,6,Footwear,Electronics,Under Armour Men's Compression EV SL Slide,view,206.56.112.1,False
4,2017-09-01 06:01:00,2017-09-01,2017,9,September,Friday,6,Fan Shop,Water Sports,Pelican Sunstream 100 Kayak,view,136.108.56.242,False


In [10]:
DROP_DUPLICATES = True

dups = int(df.duplicated().sum())
print("Duplicate rows found:", dups)

if DROP_DUPLICATES:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Removed. Rows now:", len(df))

Duplicate rows found: 3249
Removed. Rows now: 466728


In [11]:
text_cols = df.select_dtypes(include=["object", "string"]).columns

assert df.isna().sum().sum() == 0, "لسه فيه قيم فاضية"

if DROP_DUPLICATES:
    assert df.duplicated().sum() == 0, "لسه فيه صفوف مكررة"

for c in text_cols:
    assert (df[c] == df[c].str.strip()).all(), f"مسافات زيادة في {c}"
    assert not df[c].str.contains(r"\s{2,}", regex=True).any(), f"مسافات مزدوجة في {c}"
    assert not df[c].str.contains(r"[^\x20-\x7E]", regex=True).any(), f"حروف غريبة أو رموز مخفية في {c}"

assert pd.api.types.is_datetime64_any_dtype(df["timestamp"])
assert df["timestamp"].is_monotonic_increasing, "الصفوف مش مترتبة بالوقت"
assert df["timestamp"].between("2017-09-01", "2018-02-01").all(), "تاريخ برة النطاق المتوقع"
assert df["hour"].between(0, 23).all()

assert set(df["action"]) == {"view", "add_to_cart"}
assert df["is_private_ip"].dtype == bool

assert df.groupby("product")["category"].nunique().max() == 1, "منتج في أكتر من فئة"
assert df.groupby("product")["department"].nunique().max() == 1, "منتج في أكتر من قسم"

print("✅ كل الفحوصات عدّت — الداتا نضيفة 100%")

✅ كل الفحوصات عدّت — الداتا نضيفة 100%


In [12]:
print(f"Rows   : {rows_before:,} → {len(df):,}")
print(f"Columns: {cols_before} → {df.shape[1]}")
print(f"Nulls  : {int(df.isna().sum().sum())}")
print(f"Dupes  : {int(df.duplicated().sum())}")
print(f"Products: {df['product'].nunique()} | Categories: {df['category'].nunique()} | Departments: {df['department'].nunique()}")
print()
df.info()

Rows   : 469,977 → 466,728
Columns: 8 → 13
Nulls  : 0
Dupes  : 0
Products: 76 | Categories: 33 | Departments: 6

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 466728 entries, 0 to 466727
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   timestamp      466728 non-null  datetime64[ns]
 1   date           466728 non-null  datetime64[ns]
 2   year           466728 non-null  int32         
 3   month_num      466728 non-null  int32         
 4   month_name     466728 non-null  object        
 5   day_name       466728 non-null  object        
 6   hour           466728 non-null  int64         
 7   department     466728 non-null  object        
 8   category       466728 non-null  object        
 9   product        466728 non-null  object        
 10  action         466728 non-null  object        
 11  ip             466728 non-null  object        
 12  is_private_ip  466728 non-null  bool       

In [13]:
OUTPUT_PATH = OUTPUT_DIR / "tokenized_access_logs_cleaned.xlsx"
df.to_excel(OUTPUT_PATH, index=False, sheet_name="access_logs_cleaned", engine=EXCEL_ENGINE)
print("Saved:", OUTPUT_PATH, "→", df.shape)

df.head()

Saved: C:\Users\LAP TECH\Desktop\DEPI final Project\files\tokenized_access_logs_cleaned.xlsx → (466728, 13)


,timestamp,date,year,month_num,month_name,day_name,hour,department,category,product,action,ip,is_private_ip
0,2017-09-01 06:00:00,2017-09-01,2017,9,September,Friday,6,Fitness,Baseball & Softball,adidas Brazuca 2017 Official Match Ball,view,37.97.182.65,False
1,2017-09-01 06:00:00,2017-09-01,2017,9,September,Friday,6,Fan Shop,Hunting & Shooting,The North Face Women's Recon Backpack,view,206.56.112.1,False
2,2017-09-01 06:00:00,2017-09-01,2017,9,September,Friday,6,Apparel,Featured Shops,adidas Kids' RG III Mid Football Cleat,view,215.143.180.0,False
3,2017-09-01 06:00:00,2017-09-01,2017,9,September,Friday,6,Footwear,Electronics,Under Armour Men's Compression EV SL Slide,view,206.56.112.1,False
4,2017-09-01 06:01:00,2017-09-01,2017,9,September,Friday,6,Fan Shop,Water Sports,Pelican Sunstream 100 Kayak,view,136.108.56.242,False
